SHAP explanation (GradientExplainer)

In [ ]:
try:
    import shap
    shap.initjs()

    def sample_background_preprocessed(generator, k=20):
        files = generator.filepaths
        k = min(k, len(files))
        rng = np.random.RandomState(42)
        idxs = rng.choice(len(files), size=k, replace=False)
        imgs = []
        for i in idxs:
            bgr = cv2.imread(files[i])
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
            pre = densenet_preprocess(rgb.astype(np.float32))
            imgs.append(pre)
        return np.stack(imgs, axis=0)

    background_pre = sample_background_preprocessed(train_gen, k=20)
    explainer = shap.GradientExplainer(model, background_pre)

    print("Computing SHAP values for selected image (may take some time)...")
    shap_values = explainer.shap_values(inp_batch)  # list or array

    if isinstance(shap_values, list):
        sv_to_plot = shap_values[0]
    else:
        sv_to_plot = shap_values

    try:
        shap.image_plot(sv_to_plot, inp_batch)
    except Exception as e:
        # fallback: heatmap of mean |SHAP|
        arr = np.abs(sv_to_plot[0]).mean(axis=-1)
        plt.figure(figsize=(8,4))
        plt.subplot(1,2,1); plt.title("Original"); plt.imshow(orig_rgb_resized); plt.axis('off')
        plt.subplot(1,2,2); plt.title("SHAP (avg |value|)"); plt.imshow(arr, cmap='hot'); plt.axis('off')
        plt.tight_layout()
        plt.show()

    print("SHAP explanation done.")
except Exception as e:
    print("SHAP failed or not installed. Error:", e)
